# Venezuela earthquake (2026-06-24) — before/after imagery comparison

Compare **pre-earthquake high-resolution imagery** (Esri Wayback, 2026-05-28 release, sub-meter) against **post-earthquake SkySat/Pelican imagery (50 cm, 26–28 June 2026)** to find evidence of landslides and flag areas for field investigation. A terrain hillshade is one click away for reading slopes and drainages.

- Post-event data: [source.coop/planet/venezuela-earthquake-2026-06-24](https://source.coop/planet/venezuela-earthquake-2026-06-24) — imagery © Planet Labs PBC, **CC BY-NC 4.0**
- Pre-event: [Esri World Imagery Wayback](https://livingatlas.arcgis.com/wayback/) — **capture dates vary by tile**; check the Wayback site before citing a "before" date for a specific slope
- Nothing is downloaded: maps stream tiles from the cloud (needs internet)
- Run cells top to bottom. Change `LOCATION` / `SCENE` and re-run from there to switch areas.

In [ ]:
import leafmap

from geer_venezuela import (
    ATTRIBUTION,
    HILLSHADE,
    HILLSHADE_ATTRIBUTION,
    WAYBACK_ATTRIBUTION,
    WAYBACK_PRE_EVENT,
    add_compare_control,
    add_tagging_control,
    asset_href,
    load_items,
    locations,
    save_tagged_features,
    scenes_for,
)

items = load_items("post-event")
locations(items)

## 1. Overview — where the post-event imagery is

Red outlines are post-event scene footprints over the terrain hillshade. Click a footprint for the scene id and acquisition time.

In [ ]:
overview = leafmap.Map(center=(10.5, -67.4), zoom=8)
overview.add_tile_layer(
    HILLSHADE,
    name="Terrain hillshade",
    attribution=HILLSHADE_ATTRIBUTION,
)
footprints = items[
    ["id", "title", "location", "datetime", "constellation", "eo:cloud_cover", "geometry"]
].assign(datetime=lambda d: d["datetime"].astype(str))
overview.add_gdf(
    footprints,
    layer_name="Post-event scene footprints",
    style={"color": "#ff3b30", "weight": 2, "fillOpacity": 0.05},
    zoom_to_layer=False,
)
overview

## 2. Pick a location and scene

Set `LOCATION` to one of: `caracas`, `catia-la-mar`, `independencia-ocumare`, `la-guaira`, `puerto-cabello`, `valencia`, `yumare`; `SCENE` is a row number from the table (prefer low `eo:cloud_cover`).

In [ ]:
LOCATION = "la-guaira"
SCENE = 0

scenes = scenes_for(items, LOCATION)
scene = scenes.iloc[SCENE]
scenes[["id", "datetime", "constellation", "gsd", "eo:cloud_cover", "title"]]

## 3. Compare — BEFORE / AFTER / TOPO

One map, three views, switched by the **button bar pinned to the top-left**:

- **BEFORE** — Esri Wayback 2026-05-28 (sub-meter)
- **AFTER** — post-event 50 cm scene
- **TOPO** — terrain hillshade, for reading slopes, drainages, and runout paths

Click BEFORE/AFTER repeatedly to flicker — new landslide scars pop out. Flip to TOPO to see whether a scar sits on a steep face or above a channel.

In [ ]:
m = leafmap.Map()
m.add_tile_layer(
    HILLSHADE,
    name="TOPO — terrain hillshade",
    attribution=HILLSHADE_ATTRIBUTION,
)
m.add_tile_layer(
    WAYBACK_PRE_EVENT,
    name="BEFORE — Esri Wayback 2026-05-28",
    attribution=WAYBACK_ATTRIBUTION,
    max_zoom=19,
)
m.add_cog_layer(
    asset_href(scene, "visual"),
    name=f"AFTER — {scene['title']}",
    attribution=ATTRIBUTION,
)
add_compare_control(
    m,
    {
        "BEFORE": "BEFORE — Esri Wayback 2026-05-28",
        "AFTER": f"AFTER — {scene['title']}",
        "TOPO": "TOPO — terrain hillshade",
    },
    selected="AFTER",
)
add_tagging_control(m)
m

## 4. Mark and tag landslide candidates

The **tag panel (top-right of the map)** sets the category and an optional note; every feature you draw gets stamped with the panel's *current* values — so set the tag first, then draw. The counter shows how many features you've tagged this session; deleting a feature with the draw tools untags it too. Then run the next cell to export GeoJSON (loads in QGIS, ArcGIS Online, Google Earth, etc.).

In [ ]:
from pathlib import Path

out_dir = Path.cwd().parent / "data" / "landslide_candidates"
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / f"{LOCATION}_{scene['id']}.geojson"

n = save_tagged_features(m, out_file)
if n:
    print(f"Saved {n} tagged feature(s) to {out_file}")
else:
    print("Nothing drawn yet — set a tag, draw on the map above, then re-run this cell.")

## What to look for

- **Fresh scars**: light-toned (tan/grey) patches of bare soil/rock on vegetated slopes that are absent in the BEFORE image.
- **Debris runouts**: fans or flow paths below scars — down drainages, across roads, into buildings (trace the channel in TOPO view).
- **Blocked drainages / turbid water**: sediment plumes at river mouths, ponding upstream of debris.
- **Road cuts and coastal bluffs**: common failure points; scan along the Caracas–La Guaira corridor.

Caveats:

- Wayback capture dates vary by tile and can predate the release by months–years — verify at the [Wayback site](https://livingatlas.arcgis.com/wayback/) before citing a "before" date.
- Check `eo:cloud_cover` and prefer clear scenes; the `udm2` asset of each scene is a per-pixel cloud/shadow mask if needed.

## Ideas for additional data (later)

- **Sentinel-2** (10 m, free, ~5-day revisit) via Earth Search STAC — regional sweep beyond the Planet footprints.
- **Maxar Open Data Program** — often releases 30–50 cm imagery for major disasters (not activated for this event as of 2026-07-03).
- **Copernicus EMS / UNOSAT** rapid-mapping activations — may already have damage/landslide vectors.
- **NASA/USGS**: ShakeMap + slope data to prioritize where landslides are *likely*, not just visible.